# 08 - Otimizacao ONNX

Objetivo: validar a Etapa 4 do Tech Challenge comparando o modelo servido em `.pkl` com a versao exportada para ONNX Runtime.

Este notebook nao escolhe um novo modelo. Ele otimiza o artefato definido em `configs/model_config.yaml` (`tfidf_logreg`) porque esse e o modelo realmente servido pela API e preserva `predict_proba`.

In [ ]:
import pandas as pd

from src.config import MODELS_DIR, load_config
from src.optimization.onnx import compare_artifacts, export_model

model_name = load_config()["serving"]["model"]
model_dir = MODELS_DIR / model_name
model_dir

## Exportacao

A conversao remove `lowercase` e `strip_accents` do grafo ONNX. Essa normalizacao passa a ocorrer em Python antes da chamada ao ONNX Runtime, evitando incompatibilidades conhecidas do `skl2onnx` com `strip_accents="unicode"` e problemas de locale em imagens Linux slim.

In [ ]:
metadata = export_model(model_dir)
metadata

## Equivalencia e Latencia

A comparacao usa o mesmo split de teste, o mesmo protocolo de warm-up e chamadas single-sample descrito em `docs/NOTEBOOKS.md`. Pequenas divergencias de classe podem ocorrer em exemplos exatamente na fronteira por diferenca numerica entre scikit-learn e ONNX Runtime; o limite aceito fica em `configs/model_config.yaml`.

In [ ]:
comparison = compare_artifacts(model_dir)
comparison["mismatch_rate"], comparison["mismatches"], comparison["checked_predictions"]

In [ ]:
pd.read_csv(model_dir / "latency_comparison.csv")